# Phase 0 - Environment and Project Setup

Project: ComplaintIQ

Work through this notebook incrementally. Validate each step before proceeding.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path.home() / 'complaintiq'
print('Project root:', PROJECT_ROOT)
print('Exists:', PROJECT_ROOT.exists())

Project root: /home/sagemaker-user/complaintiq
Exists: True


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.home() / "complaintiq"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    ROOT,
    REGION,
    ACCOUNT,
    BUCKET,
    CFPB_RAW,
    ECFR_RAW,
    PDF_RAW,
    PROC,
    INDEX,
    STATE,
    REPORTS,
    LOGS
)

In [3]:
print("Project root :", ROOT)
print("AWS region   :", REGION)
print("AWS account  :", ACCOUNT)
print("S3 bucket    :", BUCKET)
print("Raw CFPB     :", CFPB_RAW)
print("Raw eCFR     :", ECFR_RAW)
print("Raw PDF      :", PDF_RAW)
print("Processed    :", PROC)
print("Index        :", INDEX)
print("State        :", STATE)
print("Reports      :", REPORTS)

assert REGION == "us-east-1"
assert len(ACCOUNT) == 12
assert BUCKET == f"sagemaker-complaintiq-{ACCOUNT}"

assert all(
    path.exists()
    for path in [
        CFPB_RAW,
        ECFR_RAW,
        PDF_RAW,
        PROC,
        INDEX,
        STATE,
        REPORTS,
        LOGS
    ]
)

print("\nPASS: ComplaintIQ configuration validated.")

Project root : /home/sagemaker-user/complaintiq
AWS region   : us-east-1
AWS account  : 456427397380
S3 bucket    : sagemaker-complaintiq-456427397380
Raw CFPB     : /home/sagemaker-user/complaintiq/data/raw/cfpb
Raw eCFR     : /home/sagemaker-user/complaintiq/data/raw/ecfr
Raw PDF      : /home/sagemaker-user/complaintiq/data/raw/pdf
Processed    : /home/sagemaker-user/complaintiq/data/processed
Index        : /home/sagemaker-user/complaintiq/index
State        : /home/sagemaker-user/complaintiq/state
Reports      : /home/sagemaker-user/complaintiq/reports

PASS: ComplaintIQ configuration validated.


In [4]:
import boto3
from botocore.exceptions import ClientError
from src.config import REGION, BUCKET
s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_bucket(Bucket=BUCKET)
    print("bucket exists:", BUCKET)
except ClientError:
    s3.create_bucket(Bucket=BUCKET)        # us-east-1: do NOT pass a LocationConstraint
    print("created:", BUCKET)

bucket exists: sagemaker-complaintiq-456427397380


In [5]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.home() / "complaintiq"

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Current directory:", Path.cwd())
print("Project exists:", PROJECT_ROOT.exists())
print("Python executable:", sys.executable)
print("Python version:", sys.version)

Current directory: /home/sagemaker-user/complaintiq
Project exists: True
Python executable: /opt/conda/bin/python
Python version: 3.12.15 (main, Oct  6 2026, 23:50:18) [GCC 15.3.0]


In [6]:
import importlib.metadata as metadata

packages = [
    "boto3",
    "botocore",
    "requests",
    "pandas",
    "numpy",
    "duckdb",
    "faiss-cpu",
    "rank-bm25",
    "sqlglot",
    "langgraph",
    "langchain",
    "langchain-aws",
    "streamlit",
    "pypdf",
    "tiktoken",
    "sentence-transformers",
]

print("PACKAGE VALIDATION\n")

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package:25} {version}")
    except metadata.PackageNotFoundError:
        print(f"{package:25} NOT INSTALLED")

PACKAGE VALIDATION

boto3                     1.43.106
botocore                  1.43.106
requests                  2.34.2
pandas                    2.3.3
numpy                     1.26.4
duckdb                    1.5.1
faiss-cpu                 NOT INSTALLED
rank-bm25                 NOT INSTALLED
sqlglot                   30.23.0
langgraph                 NOT INSTALLED
langchain                 0.3.27
langchain-aws             0.2.35
streamlit                 NOT INSTALLED
pypdf                     NOT INSTALLED
tiktoken                  0.14.0
sentence-transformers     NOT INSTALLED


In [7]:
%pip install -q "langgraph>=1.0" langgraph-checkpoint-sqlite "pydantic>=2" boto3 requests \
    duckdb faiss-cpu rank-bm25 pypdf lxml sqlglot numpy pandas scikit-learn tenacity streamlit sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.9.0 which is incompatible.
autogluon-multimodal 1.5.0 requires transformers[sentencepiece]<4.58,>=4.51.0, but you have transformers 5.22.1 which is incompatible.
autogluon-timeseries 1.5.0 requires transformers[sentencepiece]<4.58,>=4.51.0, but you have transformers 5.22.1 which is incompatible.
datasets 2.2.1 requires huggingface-hub<1.0.0,>=0.1.0, but you have huggingface-hub 2.4.0 which is incompat

In [14]:
%pip show langgraph langchain-core langchain langchain-aws langchain-openai

Name: langgraph
Version: 1.2.20
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License-Expression: MIT
Location: /opt/conda/lib/python3.12/site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: 
---
Name: langchain-core
Version: 1.7.6
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /opt/conda/lib/python3.12/site-packages
Requires: certifi, httpx, jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-aws, langchain-openai, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk
---
Name: langchain
Version: 0.3.27
Summary: Building applications with LLMs through composability
Home-page:

In [3]:
import sys
from pathlib import Path

# --------------------------------------------------
# 1. Configure project root
# --------------------------------------------------

PROJECT_ROOT = Path.home() / "complaintiq"

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"ComplaintIQ project directory not found: {PROJECT_ROOT}"
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --------------------------------------------------
# 2. Import project configuration
# --------------------------------------------------

from src.config import (
    ROOT,
    REGION,
    ACCOUNT,
    BUCKET,
    CFPB_RAW,
    ECFR_RAW,
    PDF_RAW,
    PROC,
    INDEX,
    STATE,
    REPORTS,
    LOGS,
)

# --------------------------------------------------
# 3. Display configuration
# --------------------------------------------------

print("=" * 60)
print("COMPLAINTIQ CONFIGURATION VALIDATION")
print("=" * 60)

print(f"PROJECT ROOT : {ROOT}")
print(f"AWS REGION   : {REGION}")
print(f"AWS ACCOUNT  : {ACCOUNT}")
print(f"S3 BUCKET    : {BUCKET}")

# --------------------------------------------------
# 4. Validate AWS configuration
# --------------------------------------------------

assert ROOT.resolve() == PROJECT_ROOT.resolve(), "Incorrect project root"
assert REGION == "us-east-1", "Incorrect AWS region"
assert len(ACCOUNT) == 12, "Invalid AWS account ID"
assert BUCKET == f"sagemaker-complaintiq-{ACCOUNT}", "Incorrect S3 bucket"

print("\nPASS: AWS configuration validated.")

# --------------------------------------------------
# 5. Validate project directories
# --------------------------------------------------

directories = {
    "CFPB RAW": CFPB_RAW,
    "ECFR RAW": ECFR_RAW,
    "PDF RAW": PDF_RAW,
    "PROCESSED": PROC,
    "INDEX": INDEX,
    "STATE": STATE,
    "REPORTS": REPORTS,
    "LOGS": LOGS,
}

print("\nDIRECTORY VALIDATION")
print("-" * 60)

for name, directory in directories.items():
    status = "PASS" if Path(directory).is_dir() else "FAIL"
    print(f"{name:<15}: {directory} [{status}]")

# --------------------------------------------------
# 6. Final validation
# --------------------------------------------------

missing_directories = [
    name
    for name, directory in directories.items()
    if not Path(directory).is_dir()
]

assert not missing_directories, (
    f"Missing project directories: {missing_directories}"
)

print("\n" + "=" * 60)
print("PASS: ALL COMPLAINTIQ CONFIGURATIONS VALIDATED")
print("=" * 60)

COMPLAINTIQ CONFIGURATION VALIDATION
PROJECT ROOT : /home/sagemaker-user/complaintiq
AWS REGION   : us-east-1
AWS ACCOUNT  : 456427397380
S3 BUCKET    : sagemaker-complaintiq-456427397380

PASS: AWS configuration validated.

DIRECTORY VALIDATION
------------------------------------------------------------
CFPB RAW       : /home/sagemaker-user/complaintiq/data/raw/cfpb [PASS]
ECFR RAW       : /home/sagemaker-user/complaintiq/data/raw/ecfr [PASS]
PDF RAW        : /home/sagemaker-user/complaintiq/data/raw/pdf [PASS]
PROCESSED      : /home/sagemaker-user/complaintiq/data/processed [PASS]
INDEX          : /home/sagemaker-user/complaintiq/index [PASS]
STATE          : /home/sagemaker-user/complaintiq/state [PASS]
REPORTS        : /home/sagemaker-user/complaintiq/reports [PASS]
LOGS           : /home/sagemaker-user/complaintiq/logs [PASS]

PASS: ALL COMPLAINTIQ CONFIGURATIONS VALIDATED


In [6]:
import importlib.metadata as metadata

packages = [
    "boto3",
    "botocore",
    "requests",
    "pandas",
    "numpy",
    "duckdb",
    "faiss-cpu",
    "rank-bm25",
    "sqlglot",
    "langgraph",
    "langchain",
    "langchain-aws",
    "streamlit",
    "pypdf",
    "tiktoken",
    "sentence-transformers",
]

for package in packages:
    try:
        print(f"{package:25} {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:25} NOT INSTALLED")

boto3                     1.43.106
botocore                  1.43.106
requests                  2.34.2
pandas                    2.3.3
numpy                     1.26.4
duckdb                    1.5.1
faiss-cpu                 1.15.1
rank-bm25                 0.2.2
sqlglot                   30.23.0
langgraph                 1.2.20
langchain                 0.3.27
langchain-aws             0.2.35
streamlit                 1.66.1
pypdf                     6.21.4
tiktoken                  0.14.0
sentence-transformers     6.2.0


In [1]:
from pathlib import Path

requirements = """
boto3==1.43.106
botocore==1.43.106
requests==2.34.2
pandas==2.3.3
numpy==1.26.4
duckdb==1.5.1
faiss-cpu==1.15.1
rank-bm25==0.2.2
sqlglot==30.23.0
langgraph==1.2.20
langchain==0.3.27
langchain-aws==0.2.35
streamlit==1.66.1
pypdf==6.21.4
tiktoken==0.14.0
sentence-transformers==6.2.0
""".strip()

requirements_path = Path.home() / "complaintiq" / "requirements.txt"
requirements_path.write_text(requirements + "\n", encoding="utf-8")

print("Created:", requirements_path)
print(requirements_path.read_text())

Created: /home/sagemaker-user/complaintiq/requirements.txt
boto3==1.43.106
botocore==1.43.106
requests==2.34.2
pandas==2.3.3
numpy==1.26.4
duckdb==1.5.1
faiss-cpu==1.15.1
rank-bm25==0.2.2
sqlglot==30.23.0
langgraph==1.2.20
langchain==0.3.27
langchain-aws==0.2.35
streamlit==1.66.1
pypdf==6.21.4
tiktoken==0.14.0
sentence-transformers==6.2.0



In [3]:
%pip uninstall -y sentence-transformers transformers datasets

Found existing installation: sentence-transformers 6.2.0
Uninstalling sentence-transformers-6.2.0:
  Successfully uninstalled sentence-transformers-6.2.0
Found existing installation: transformers 5.22.1
Uninstalling transformers-5.22.1:
  Successfully uninstalled transformers-5.22.1
Found existing installation: datasets 2.2.1
Uninstalling datasets-2.2.1:
  Successfully uninstalled datasets-2.2.1
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path

requirements = """
boto3==1.43.106
botocore==1.43.106
requests==2.34.2
pandas==2.3.3
numpy==1.26.4
duckdb==1.5.1
faiss-cpu==1.15.1
rank-bm25==0.2.2
sqlglot==30.23.0
langgraph==1.2.20
langchain==0.3.27
langchain-aws==0.2.35
streamlit==1.66.1
pypdf==6.21.4
tiktoken==0.14.0
sentence-transformers==6.2.0
""".strip()

requirements_path = Path.home() / "complaintiq" / "requirements.txt"
requirements_path.write_text(requirements + "\n", encoding="utf-8")

print("Created:", requirements_path)
print(requirements_path.read_text())

Created: /home/sagemaker-user/complaintiq/requirements.txt
boto3==1.43.106
botocore==1.43.106
requests==2.34.2
pandas==2.3.3
numpy==1.26.4
duckdb==1.5.1
faiss-cpu==1.15.1
rank-bm25==0.2.2
sqlglot==30.23.0
langgraph==1.2.20
langchain==0.3.27
langchain-aws==0.2.35
streamlit==1.66.1
pypdf==6.21.4
tiktoken==0.14.0
sentence-transformers==6.2.0



In [2]:
import boto3
import requests
import pandas
import numpy
import duckdb
import faiss

from rank_bm25 import BM25Okapi
import sqlglot
import langgraph
import streamlit
import pypdf
import tiktoken

from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel

print("PASS: All ComplaintIQ core package imports succeeded.")

PASS: All ComplaintIQ core package imports succeeded.


In [2]:
import boto3

REGION = "us-east-1"

bedrock = boto3.client(
    "bedrock",
    region_name=REGION
)

bedrock_runtime = boto3.client(
    "bedrock-runtime",
    region_name=REGION
)

print("Bedrock client initialized.")
print("Region:", REGION)

Bedrock client initialized.
Region: us-east-1


In [3]:
import json

SONNET_MODEL = "us.anthropic.claude-sonnet-4-6"

response = bedrock_runtime.converse(
    modelId=SONNET_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Reply with exactly: ComplaintIQ Sonnet test successful."
                }
            ]
        }
    ],
    inferenceConfig={
        "maxTokens": 100,
        "temperature": 0
    }
)

sonnet_output = "".join(
    item["text"]
    for item in response["output"]["message"]["content"]
    if "text" in item
)

print("Model:", SONNET_MODEL)
print("Response:", sonnet_output)
print("Stop reason:", response.get("stopReason"))
print("\nPASS: Sonnet invocation successful.")

Model: us.anthropic.claude-sonnet-4-6
Response: ComplaintIQ Sonnet test successful.
Stop reason: end_turn

PASS: Sonnet invocation successful.


In [4]:
OPUS_MODEL = "us.anthropic.claude-opus-4-8"

response = bedrock_runtime.converse(
    modelId=OPUS_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": "Reply with exactly: ComplaintIQ Opus test successful."
                }
            ]
        }
    ],
    inferenceConfig={
        "maxTokens": 100
    }
)

opus_output = "".join(
    item["text"]
    for item in response["output"]["message"]["content"]
    if "text" in item
)

print("Model:", OPUS_MODEL)
print("Response:", opus_output)
print("Stop reason:", response.get("stopReason"))
print("\nPASS: Opus invocation successful.")

Model: us.anthropic.claude-opus-4-8
Response: ComplaintIQ Opus test successful.
Stop reason: end_turn

PASS: Opus invocation successful.


In [5]:
import json
import numpy as np

EMBED_MODEL = "amazon.titan-embed-text-v2:0"

text = "A customer reported unauthorized transactions on their savings account."

response = bedrock_runtime.invoke_model(
    modelId=EMBED_MODEL,
    body=json.dumps({
        "inputText": text,
        "dimensions": 1024,
        "normalize": True
    }),
    contentType="application/json",
    accept="application/json"
)

embedding_result = json.loads(
    response["body"].read()
)

embedding = np.array(
    embedding_result["embedding"],
    dtype=np.float32
)

print("Model:", EMBED_MODEL)
print("Embedding shape:", embedding.shape)
print("Embedding norm:", np.linalg.norm(embedding))
print("All finite:", np.isfinite(embedding).all())

assert embedding.shape == (1024,)
assert np.isfinite(embedding).all()
assert abs(np.linalg.norm(embedding) - 1.0) < 0.001

print("\nPASS: Titan embedding validation successful.")

Model: amazon.titan-embed-text-v2:0
Embedding shape: (1024,)
Embedding norm: 1.0
All finite: True

PASS: Titan embedding validation successful.


In [8]:

import json
import boto3

from botocore.config import Config
from botocore.exceptions import ClientError, BotoCoreError

from src.config import REGION, ROOT


# --------------------------------------------------
# 1. Initialize Bedrock Runtime
# --------------------------------------------------

brt = boto3.client(
    "bedrock-runtime",
    region_name=REGION,
    config=Config(
        retries={
            "max_attempts": 10,
            "mode": "adaptive"
        },
        read_timeout=120,
        connect_timeout=10
    )
)


# --------------------------------------------------
# 2. Candidate Models
# --------------------------------------------------

CANDIDATES = {
    "worker": [
        "anthropic.claude-sonnet-4-6",
        "us.anthropic.claude-sonnet-4-6",
        "global.anthropic.claude-sonnet-4-6"
    ],

    "judge": [
        "anthropic.claude-opus-4-8",
        "us.anthropic.claude-opus-4-8",
        "global.anthropic.claude-opus-4-8"
    ],

    "fast": [
        "anthropic.claude-3-haiku-20240307-v1:0",
        "us.anthropic.claude-3-haiku-20240307-v1:0"
    ]
}

EMBED_MODEL = "amazon.titan-embed-text-v2:0"


# --------------------------------------------------
# 3. Probe Text Models
# --------------------------------------------------

def probe(model_id):

    try:

        response = brt.converse(
            modelId=model_id,
            messages=[
                {
                    "role": "user",
                    "content": [{"text": "ping"}]
                }
            ],
            inferenceConfig={
                "maxTokens": 5
            }
        )

        return True, "OK"

    except ClientError as e:

        err = e.response.get("Error", {})

        return False, (
            f'{err.get("Code", "Unknown")}: '
            f'{err.get("Message", "")[:150]}'
        )

    except BotoCoreError as e:

        return False, f"AWS SDK Error: {str(e)[:150]}"

    except Exception as e:

        return False, f"Unexpected Error: {str(e)[:150]}"


# --------------------------------------------------
# 4. Resolve Models
# --------------------------------------------------

MODELS = {}

for role, ids in CANDIDATES.items():

    print(f"\nChecking {role.upper()} models")

    for mid in ids:

        ok, msg = probe(mid)

        print(f"{mid:48s} {msg}")

        if ok:

            MODELS[role] = mid

            print(f"Selected {role}: {mid}")

            break


# --------------------------------------------------
# 5. Fallback Handling
# --------------------------------------------------

if not MODELS.get("worker"):

    raise RuntimeError(
        "No Claude worker model resolved. "
        "Check Bedrock model access and IAM permissions."
    )

if not MODELS.get("fast"):

    MODELS["fast"] = MODELS["worker"]

    print("WARNING: Fast model unavailable. "
          "Using worker model instead.")

if not MODELS.get("judge"):

    MODELS["judge"] = MODELS["worker"]

    print("WARNING: Judge model unavailable. "
          "Using worker model instead.")


# --------------------------------------------------
# 6. Verify Embedding Model
# --------------------------------------------------

def probe_embedding(model_id):

    try:

        response = brt.invoke_model(
            modelId=model_id,
            contentType="application/json",
            accept="application/json",
            body=json.dumps({
                "inputText": "embedding connectivity test"
            })
        )

        result = json.loads(
            response["body"].read()
        )

        if not result.get("embedding"):

            return False, "Embedding response missing vector"

        return True, "OK"

    except ClientError as e:

        err = e.response.get("Error", {})

        return False, (
            f'{err.get("Code", "Unknown")}: '
            f'{err.get("Message", "")[:150]}'
        )

    except Exception as e:

        return False, str(e)[:150]


embed_ok, embed_msg = probe_embedding(EMBED_MODEL)

print(f"\nEmbedding model: {EMBED_MODEL}")
print(f"Embedding status: {embed_msg}")

if not embed_ok:

    raise RuntimeError(
        "Titan embedding model unavailable. "
        "Check Bedrock access and permissions."
    )

MODELS["embed"] = EMBED_MODEL


# --------------------------------------------------
# 7. Save Configuration
# --------------------------------------------------

output_file = ROOT / "models.json"

output_file.write_text(
    json.dumps(MODELS, indent=2)
)

print("\n========== FINAL MODEL CONFIGURATION ==========")

print(json.dumps(MODELS, indent=2))

print(f"\nSaved to: {output_file}")



Checking WORKER models
anthropic.claude-sonnet-4-6                      ValidationException: Invocation of model ID anthropic.claude-sonnet-4-6 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference pr
us.anthropic.claude-sonnet-4-6                   OK
Selected worker: us.anthropic.claude-sonnet-4-6

Checking JUDGE models
anthropic.claude-opus-4-8                        ValidationException: Invocation of model ID anthropic.claude-opus-4-8 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference prof
us.anthropic.claude-opus-4-8                     OK
Selected judge: us.anthropic.claude-opus-4-8

Checking FAST models
anthropic.claude-3-haiku-20240307-v1:0           ResourceNotFoundException: This model version has reached the end of its life. Please refer to the AWS documentation for more details.
us.anthropic.claude-3-haiku-20240307-v1:0        ResourceNotFoundException: This model version has reached t

In [7]:
from importlib.metadata import version, PackageNotFoundError

required_packages = [
    "boto3",
    "botocore",
    "requests",
    "pandas",
    "numpy",
    "duckdb",
    "faiss-cpu",
    "rank-bm25",
    "sqlglot",
    "langgraph",
    "langchain",
    "langchain-aws",
    "streamlit",
    "pypdf",
    "tiktoken",
    "sentence-transformers",
]

missing = []

for package in required_packages:
    try:
        print(f"{package:25} {version(package)}")
    except PackageNotFoundError:
        missing.append(package)
        print(f"{package:25} MISSING")

if missing:
    print("\nMissing packages:", missing)
else:
    print("\nPASS: All required dependencies are installed.")

boto3                     1.43.106
botocore                  1.43.106
requests                  2.34.2
pandas                    2.3.3
numpy                     1.26.4
duckdb                    1.5.1
faiss-cpu                 1.15.1
rank-bm25                 0.2.2
sqlglot                   30.23.0
langgraph                 1.2.20
langchain                 0.3.27
langchain-aws             0.2.35
streamlit                 1.66.1
pypdf                     6.21.4
tiktoken                  0.14.0
sentence-transformers     MISSING

Missing packages: ['sentence-transformers']


In [8]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Dependency conflicts detected."

print("PASS: No dependency conflicts.")

autogluon-multimodal 1.5.0 requires nvidia-ml-py3, which is not installed.
autogluon-multimodal 1.5.0 requires transformers, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting, which is not installed.
autogluon-timeseries 1.5.0 requires einops, which is not installed.
autogluon-timeseries 1.5.0 requires peft, which is not installed.
autogluon-timeseries 1.5.0 requires transformers, which is not installed.
dash 2.18.1 requires dash-core-components, which is not installed.
dash 2.18.1 requires dash-html-components, which is not installed.
dash 2.18.1 requires dash-table, which is not installed.
evaluate 0.4.1 requires datasets, which is not installed.
sagemaker-studio 1.1.35 requires pymongo-auth-aws, which is not installed.
sagemaker-studio 1.1.35 requires pymongosql, which is not installed.
skops 0.16.0 requires prettytable, which is not installed.
autogluon-common 1.5.0 has requirement pyarrow<21.0.0,>=7.0.0, but you have pyarrow 21.0.0.
autogluon-multimod

AssertionError: Dependency conflicts detected.

In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config

assert config.ROOT.exists()
assert config.CFPB_RAW.exists()
assert config.ECFR_RAW.exists()
assert config.PDF_RAW.exists()
assert config.PROC.exists()
assert config.INDEX.exists()
assert config.STATE.exists()
assert config.REPORTS.exists()
assert config.LOGS.exists()

print("Project root:", config.ROOT)
print("AWS Region:", config.REGION)
print("AWS Account:", config.ACCOUNT)
print("S3 Bucket:", config.BUCKET)

print("\nPASS: Project configuration and directory validation successful.")

Project root: /home/sagemaker-user/complaintiq
AWS Region: us-east-1
AWS Account: 456427397380
S3 Bucket: sagemaker-complaintiq-456427397380

PASS: Project configuration and directory validation successful.


In [10]:
from pathlib import Path

PROJECT_ROOT = Path("/home/sagemaker-user/complaintiq")

required_files = [
    "README.md",
    "requirements.txt",
    "models.json",
    "src/config.py",
    "src/download_data.py",
    "src/prepare.py",
    "src/indexing.py",
    "src/llm.py",
    "notebooks/01_setup.ipynb",
    "notebooks/02_data.ipynb",
    "notebooks/03_retrieval.ipynb",
    "eval/gold_triage.csv",
    "eval/gold_sql.jsonl",
    "eval/gold_regqa.jsonl",
]

missing_files = []

for filename in required_files:
    path = PROJECT_ROOT / filename

    if path.exists():
        print("FOUND :", filename)
    else:
        print("MISSING:", filename)
        missing_files.append(filename)

assert not missing_files, f"Missing project files: {missing_files}"

print("\nPASS: Required project files are present.")

FOUND : README.md
FOUND : requirements.txt
FOUND : models.json
FOUND : src/config.py
FOUND : src/download_data.py
FOUND : src/prepare.py
FOUND : src/indexing.py
FOUND : src/llm.py
FOUND : notebooks/01_setup.ipynb
FOUND : notebooks/02_data.ipynb
FOUND : notebooks/03_retrieval.ipynb
FOUND : eval/gold_triage.csv
FOUND : eval/gold_sql.jsonl
FOUND : eval/gold_regqa.jsonl

PASS: Required project files are present.
